# Candidate SSE Socio-Geodemographic Association

Lightweight runner for the main overall association analysis. See [`sse_sociodemographic_association_notes.md`](sse_sociodemographic_association_notes.md) for the model rationale, inputs, output table definitions, and interpretation guide.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection import lib as sselib  # noqa: E402

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## Configuration

In [2]:
RESULT_DIR = PROJECT_ROOT / "sse_detection" / "association_outputs"
MODEL_METHOD = "firth_glm"
VARIANT_ADJUSTER = "clade"
WINDOW_ADJUSTMENT = "fixed_effects"
MIXING_REFERENCE = "per 1 SD entropy null-model z-score"

model_sets = sselib.default_model_sets(
    variant_adjuster=VARIANT_ADJUSTER,
    window_adjustment=WINDOW_ADJUSTMENT,
)
model_sets

{'primary': ['C(window_idx)', 'C(clade)'],
 'expanded': ['C(window_idx)',
  'C(clade)',
  'z_dz_cum_prop_sequenced',
  'z_dz_cum_incidence_per_capita',
  'z_dz_7d_test_positivity',
  'z_log1p_dz_cum_positive_tests']}

## Fit and Save

In [3]:
result = sselib.run_main_association_analysis(
    project_root=PROJECT_ROOT,
    result_dir=RESULT_DIR,
    model_method=MODEL_METHOD,
    variant_adjuster=VARIANT_ADJUSTER,
    window_adjustment=WINDOW_ADJUSTMENT,
    mixing_reference=MIXING_REFERENCE,
)

summary_tables = result["summary_tables"]
composition_wald = summary_tables["composition_wald.csv"]
composition_or = summary_tables["composition_odds_ratios.csv"]
composition_fit_stats = summary_tables["composition_fit_stats.csv"]
mixing_wald = summary_tables["mixing_wald.csv"]
mixing_or = summary_tables["mixing_odds_ratios.csv"]
mixing_fit_stats = summary_tables["mixing_fit_stats.csv"]

print(f"Results saved to: {result['result_dir']}")
{name: len(table) for name, table in summary_tables.items()}

Fitted composition__primary__single__sex: 264,111 rows
Fitted composition__primary__single__age_band: 264,111 rows
Fitted composition__primary__single__simd_quintile: 264,111 rows
Fitted composition__primary__single__urban_rural_class: 264,111 rows
Fitted composition__primary__single__health_board: 264,111 rows
Fitted composition__primary__joint: 264,111 rows
Fitted composition__expanded__single__sex: 264,091 rows
Fitted composition__expanded__single__age_band: 264,091 rows
Fitted composition__expanded__single__simd_quintile: 264,091 rows
Fitted composition__expanded__single__urban_rural_class: 264,091 rows
Fitted composition__expanded__single__health_board: 264,091 rows
Fitted composition__expanded__joint: 264,091 rows
Fitted mixing__primary__single__sex_entropy_z: 12,966 nodes
Fitted mixing__primary__single__age_entropy_z: 12,966 nodes
Fitted mixing__primary__single__simd_entropy_z: 12,966 nodes
Fitted mixing__primary__single__urban_rural_entropy_z: 12,966 nodes
Fitted mixing__primar

{'composition_wald.csv': 20,
 'composition_odds_ratios.csv': 152,
 'composition_fit_stats.csv': 12,
 'mixing_wald.csv': 20,
 'mixing_odds_ratios.csv': 20,
 'mixing_fit_stats.csv': 12}

## Frame Diagnostics

In [4]:
frames = result["frames"]

display(
    pd.DataFrame(
        [
            {
                "frame": "node_stats",
                "rows": len(frames.node_stats),
                "nodes": frames.node_stats["cluster_id"].nunique(),
                "candidate_rows": int(frames.node_stats["sse_candidate"].sum()),
            },
            {
                "frame": "eligible_nodes",
                "rows": len(frames.eligible_nodes),
                "nodes": frames.eligible_nodes["cluster_id"].nunique(),
                "candidate_rows": int(frames.eligible_nodes["sse_candidate"].sum()),
            },
            {
                "frame": "composition_base",
                "rows": len(frames.composition_base),
                "nodes": frames.composition_base["cluster_id"].nunique(),
                "candidate_rows": int(frames.composition_base["candidate"].sum()),
            },
        ]
    )
)

print(f"Minimum candidate cluster size: {frames.min_candidate_size}")
display(result["cluster_diagnostics"])

,frame,rows,nodes,candidate_rows
0,node_stats,99006,99006,6907
1,eligible_nodes,13059,13059,6907
2,composition_base,264139,13059,147456


Minimum candidate cluster size: 6


,cluster_col,n_rows,n_clusters,min_rows_per_cluster,median_rows_per_cluster,outcome_positive_clusters,outcome_varying_clusters,analysis_frame
0,cluster_id,264139,13059,1,9.0,6907,0,composition
1,cluster_id,13059,13059,1,1.0,6907,0,node_mixing


## Composition Tables

In [5]:
display(sselib.select_table_columns(composition_wald, "wald"))
display(sselib.select_table_columns(composition_fit_stats, "fit_stats"))

,domain,model_set,predictor_set,predictor,label,reference,term,chi2,df,P>chi2,p_adj_bh,n_model_rows,n_sequences,n_nodes,dropped_nonvarying_rows,dropped_nonvarying_strata,dropped_nonvarying_detail
0,composition,primary,single,sex,Sex,Male,sex,2.436386,1,1.185491e-01,1.481863e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
1,composition,primary,single,age_band,Age band,30-34,age_band,22.743596,15,8.973759e-02,1.481863e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
2,composition,primary,single,simd_quintile,SIMD quintile,3,dz_simd_quintile,3.868955,4,4.240313e-01,4.240313e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
3,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,dz_urban_rural_class,91.060703,5,4.022191e-18,1.005548e-17,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
4,composition,primary,single,health_board,Health board,Greater Glasgow and Clyde,dz_health_board,119.197338,13,2.900509e-19,1.450255e-18,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
5,composition,primary,joint,sex,Sex,Male,sex,2.561533,1,1.094924e-01,1.368655e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
6,composition,primary,joint,age_band,Age band,30-34,age_band,22.667661,15,9.145547e-02,1.368655e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
7,composition,primary,joint,simd_quintile,SIMD quintile,3,simd_quintile,2.320146,4,6.771037e-01,6.771037e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
8,composition,primary,joint,urban_rural_class,Urban/rural class,Large Urban Areas,urban_rural_class,48.580552,5,2.703415e-09,6.758537e-09,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
9,composition,primary,joint,health_board,Health board,Greater Glasgow and Clyde,health_board,78.455095,13,2.152118e-11,1.076059e-10,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."


,domain,model_set,predictor_set,predictor,r2_mcfadden,converged,aic,bic_llf,log_likelihood,ll_null,n_model_rows,n_sequences,n_nodes
0,composition,primary,single,sex,NaN,True,335202.486013,NaN,-167514.243006,NaN,264111,188104,13058
1,composition,primary,single,age_band,NaN,True,335210.145935,NaN,-167504.072968,NaN,264111,188104,13058
2,composition,primary,single,simd_quintile,NaN,True,335207.052527,NaN,-167513.526263,NaN,264111,188104,13058
3,composition,primary,single,urban_rural_class,NaN,True,335121.734923,NaN,-167469.867461,NaN,264111,188104,13058
4,composition,primary,single,health_board,NaN,True,335108.982902,NaN,-167455.491451,NaN,264111,188104,13058
5,composition,primary,joint,all_composition,NaN,True,335081.949559,NaN,-167416.974780,NaN,264111,188104,13058
6,composition,expanded,single,sex,NaN,True,334141.765136,NaN,-166979.882568,NaN,264091,188091,13058
7,composition,expanded,single,age_band,NaN,True,334146.863440,NaN,-166968.431720,NaN,264091,188091,13058
8,composition,expanded,single,simd_quintile,NaN,True,334116.282066,NaN,-166964.141033,NaN,264091,188091,13058
9,composition,expanded,single,urban_rural_class,NaN,True,333773.683351,NaN,-166791.841676,NaN,264091,188091,13058


In [6]:
display(
    sselib.select_table_columns(
        composition_or,
        "odds_ratios",
        sort_by=["model_set", "predictor_set", "predictor", "p_value"],
    )
)

,domain,model_set,predictor_set,predictor,label,reference,term,estimate,std_error,p_value,odds_ratio,or_low,or_high
144,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_health_board, Treatment(reference='Greate...",0.234152,0.018333,2.342159e-37,1.263837,1.219231,1.310075
137,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_urban_rural_class, Treatment(reference='L...",0.271453,0.028612,2.372444e-21,1.311869,1.240325,1.387539
136,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_urban_rural_class, Treatment(reference='L...",0.111806,0.012382,1.718031e-19,1.118295,1.091484,1.145766
150,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_health_board, Treatment(reference='Greate...",0.157251,0.018254,7.031602e-18,1.170289,1.129159,1.212918
145,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_health_board, Treatment(reference='Greate...",0.197217,0.025785,2.035606e-14,1.218008,1.157981,1.281146
...,...,...,...,...,...,...,...,...,...,...,...,...,...
22,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.077790,0.009563,4.146589e-16,1.080896,1.060825,1.101347
23,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.131394,0.023415,2.004242e-08,1.140417,1.089264,1.193972
24,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.118337,0.025736,4.263183e-06,1.125624,1.070254,1.183859
20,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.056616,0.014960,1.540703e-04,1.058249,1.027670,1.089738


## Mixing Tables

In [7]:
display(sselib.select_table_columns(mixing_wald, "wald"))
display(sselib.select_table_columns(mixing_fit_stats, "fit_stats"))

,domain,model_set,predictor_set,predictor,label,reference,term,chi2,df,P>chi2,p_adj_bh,n_model_rows,n_nodes,dropped_nonvarying_rows,dropped_nonvarying_strata,dropped_nonvarying_detail
0,node_mixing,primary,single,sex_entropy_z,sex entropy z,per 1 SD entropy null-model z-score,sex_entropy_z,1.650174,1,0.198935,0.248669,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
1,node_mixing,primary,single,age_entropy_z,age entropy z,per 1 SD entropy null-model z-score,age_entropy_z,8.026412,1,0.004610,0.023050,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
2,node_mixing,primary,single,simd_entropy_z,simd entropy z,per 1 SD entropy null-model z-score,simd_entropy_z,2.020922,1,0.155145,0.248669,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
3,node_mixing,primary,single,urban_rural_entropy_z,urban rural entropy z,per 1 SD entropy null-model z-score,urban_rural_entropy_z,0.004173,1,0.948495,0.948495,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
4,node_mixing,primary,single,health_board_entropy_z,health board entropy z,per 1 SD entropy null-model z-score,health_board_entropy_z,1.951550,1,0.162420,0.248669,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
5,node_mixing,primary,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,sex_entropy_z,0.808100,1,0.368683,0.492475,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
6,node_mixing,primary,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,age_entropy_z,5.699678,1,0.016968,0.084840,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
7,node_mixing,primary,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,simd_entropy_z,0.369690,1,0.543173,0.543173,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
8,node_mixing,primary,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,urban_rural_entropy_z,0.726623,1,0.393980,0.492475,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
9,node_mixing,primary,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,health_board_entropy_z,0.757914,1,0.383982,0.492475,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."


,domain,model_set,predictor_set,predictor,r2_mcfadden,converged,aic,bic_llf,log_likelihood,ll_null,n_model_rows,n_nodes
0,node_mixing,primary,single,sex_entropy_z,NaN,True,17084.730562,NaN,-8455.365281,NaN,12966,12966
1,node_mixing,primary,single,age_entropy_z,NaN,True,17078.264046,NaN,-8452.132023,NaN,12966,12966
2,node_mixing,primary,single,simd_entropy_z,NaN,True,17084.434226,NaN,-8455.217113,NaN,12966,12966
3,node_mixing,primary,single,urban_rural_entropy_z,NaN,True,17086.487631,NaN,-8456.243816,NaN,12966,12966
4,node_mixing,primary,single,health_board_entropy_z,NaN,True,17084.514233,NaN,-8455.257117,NaN,12966,12966
5,node_mixing,primary,joint,all_mixing,NaN,True,17083.659581,NaN,-8450.829790,NaN,12966,12966
6,node_mixing,expanded,single,sex_entropy_z,NaN,True,16975.707451,NaN,-8396.853726,NaN,12966,12966
7,node_mixing,expanded,single,age_entropy_z,NaN,True,16964.890580,NaN,-8391.445290,NaN,12966,12966
8,node_mixing,expanded,single,simd_entropy_z,NaN,True,16971.885459,NaN,-8394.942729,NaN,12966,12966
9,node_mixing,expanded,single,urban_rural_entropy_z,NaN,True,16974.301017,NaN,-8396.150508,NaN,12966,12966


In [8]:
display(
    sselib.select_table_columns(
        mixing_or,
        "odds_ratios",
        sort_by=["model_set", "predictor_set", "predictor", "p_value"],
    )
)

,domain,model_set,predictor_set,predictor,label,reference,term,estimate,std_error,p_value,odds_ratio,or_low,or_high
18,node_mixing,expanded,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,urban_rural_entropy_z,0.043919,0.013156,0.000842,1.044898,1.018300,1.072191
16,node_mixing,expanded,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,age_entropy_z,-0.031856,0.011132,0.004216,0.968646,0.947740,0.990013
19,node_mixing,expanded,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,health_board_entropy_z,-0.019971,0.007858,0.011034,0.980227,0.965247,0.995440
17,node_mixing,expanded,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,simd_entropy_z,-0.014325,0.010564,0.175097,0.985777,0.965577,1.006400
15,node_mixing,expanded,joint,all_mixing,All mixing predictors,per 1 SD entropy null-model z-score,sex_entropy_z,-0.005901,0.012732,0.643014,0.994116,0.969616,1.019236
11,node_mixing,expanded,single,age_entropy_z,age entropy z,per 1 SD entropy null-model z-score,age_entropy_z,-0.036345,0.010662,0.000653,0.964307,0.944364,0.984671
14,node_mixing,expanded,single,health_board_entropy_z,health board entropy z,per 1 SD entropy null-model z-score,health_board_entropy_z,-0.015686,0.006682,0.018893,0.984436,0.971628,0.997413
10,node_mixing,expanded,single,sex_entropy_z,sex entropy z,per 1 SD entropy null-model z-score,sex_entropy_z,-0.012839,0.012518,0.305057,0.987243,0.963315,1.011765
12,node_mixing,expanded,single,simd_entropy_z,simd entropy z,per 1 SD entropy null-model z-score,simd_entropy_z,-0.021473,0.009757,0.027751,0.978756,0.960217,0.997653
13,node_mixing,expanded,single,urban_rural_entropy_z,urban rural entropy z,per 1 SD entropy null-model z-score,urban_rural_entropy_z,0.018250,0.011532,0.113532,1.018418,0.995657,1.041699


## Fit Failures

In [9]:
if not result["failures"].empty:
    display(result["failures"])
else:
    print("No model failures recorded.")

No model failures recorded.
